<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Youtube-Downloader/blob/main/Colab-Youtube-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Downloader

Download single videos or entire playlists to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install yt-dlp ffmpeg-python -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/YouTubeDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/YouTubeDownloader/'  # final destination

URL = ''  # YouTube video or playlist URL
FORMAT = 'bestvideo+bestaudio/best'  # format selection (see table below)
AUDIO_ONLY = False  # extract audio only (MP3)
MAX_VIDEOS = 0  # max videos for playlists (0 = all)

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil, glob
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def main():
    import yt_dlp
    from IPython.display import display, HTML, Javascript

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print("Keep-alive active")

    if not URL:
        print("ERROR: Set URL in config (e.g. URL = 'https://youtube.com/watch?v=...')")
        return

    begin = time.time()
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    ydl_opts = {
        'outtmpl': os.path.join(SAVE_PATH, '%(title)s.%(ext)s'),
        'format': FORMAT,
        'quiet': True,
        'no_warnings': True,
        'playlistend': MAX_VIDEOS if MAX_VIDEOS > 0 else None,
    }
    if AUDIO_ONLY:
        ydl_opts['format'] = 'bestaudio/best'
        ydl_opts['postprocessors'] = [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }]

    # detect playlist vs single for display
    with yt_dlp.YoutubeDL({'quiet': True, 'no_warnings': True, 'extract_flat': True}) as ydl:
        info = ydl.extract_info(URL, download=False)
    is_playlist = info.get('_type') == 'playlist' or 'entries' in info

    total = 0
    if is_playlist:
        entries = list(info.get('entries', []))
        total = len(entries)
        if MAX_VIDEOS > 0:
            total = min(total, MAX_VIDEOS)
        print(f'Playlist: {total} video(s) detected')
    else:
        print(f'Video: {info.get("title", "Unknown")}')

    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    def progress_hook(d):
        if d['status'] != 'downloading':
            return
        info_dict = d.get('info_dict', {})
        idx = info_dict.get('playlist_index', '')
        total_pl = info_dict.get('playlist_count', 0)
        pct = d.get('_percent_str', '').strip()
        speed = d.get('_speed_str', '').strip()
        downloaded = d.get('downloaded_bytes', 0)
        total_bytes = d.get('total_bytes', 0) or d.get('total_bytes_estimate', 0)
        bar_len = int(downloaded * 50 // max(total_bytes, 1)) if total_bytes else 0
        bar = '#' * bar_len + '-' * (50 - bar_len)
        msg = 'Downloading'
        if total_pl and idx:
            msg += f' ({idx}/{total_pl})'
        msg += f': |{bar}| {pct}'
        if speed:
            msg += f' @ {speed}'
        progress_display.update(HTML(f'<pre>{msg}</pre>'))

    ydl_opts['progress_hooks'] = [progress_hook]
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([URL])

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')

    items = glob.glob(os.path.join(SAVE_PATH, '*'))
    items = [f for f in items if os.path.isfile(f)]

    if len(items) > 1:
        import zipfile
        total_sz = sum(os.path.getsize(f) for f in items)
        processed = 0
        zpath = os.path.join(os.path.dirname(SAVE_PATH.strip('/\\')), 'YouTubeDownloader.zip')
        print(f'\nZipping {len(items)} files ({format_bytes(total_sz)})...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in items:
                zf.write(fpath, os.path.basename(fpath))
                processed += os.path.getsize(fpath)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {os.path.basename(fpath)}</pre>'))
        print(f'Zip: {os.path.basename(zpath)} ({format_bytes(os.path.getsize(zpath))})')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')
    for item in sorted(os.listdir(DRIVE_PATH)):
        print(f'  {item}')
    print('=' * 50)

if __name__ == '__main__':
    main()